# Part 8: Time Series Analysis

[← Back to Index](Index.ipynb)

**Quick Reference Guide for Time Series Forecasting**

---
## 8.1 Time Series Fundamentals

**Key Components:**
1. **Trend:** Long-term increase/decrease
2. **Seasonality:** Regular periodic fluctuations
3. **Cyclical:** Long-term oscillations (no fixed period)
4. **Noise:** Random variations

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Create sample time series
np.random.seed(42)
dates = pd.date_range('2020-01-01', periods=365, freq='D')

# Trend + Seasonality + Noise
trend = np.linspace(100, 200, 365)
seasonal = 20 * np.sin(2 * np.pi * np.arange(365) / 365)
noise = np.random.randn(365) * 5
values = trend + seasonal + noise

ts = pd.Series(values, index=dates)

# Plot
plt.figure(figsize=(14, 6))
plt.plot(ts)
plt.title('Time Series with Trend, Seasonality, and Noise')
plt.xlabel('Date')
plt.ylabel('Value')
plt.grid(True)
plt.show()

### Time Series Decomposition

**Purpose:** Separate components (trend, seasonal, residual)

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose

# Decomposition (additive or multiplicative)
decomposition = seasonal_decompose(ts, model='additive', period=30)

# Plot components
fig, axes = plt.subplots(4, 1, figsize=(14, 10))

decomposition.observed.plot(ax=axes[0], title='Original')
axes[0].set_ylabel('Observed')

decomposition.trend.plot(ax=axes[1], title='Trend')
axes[1].set_ylabel('Trend')

decomposition.seasonal.plot(ax=axes[2], title='Seasonal')
axes[2].set_ylabel('Seasonal')

decomposition.resid.plot(ax=axes[3], title='Residual')
axes[3].set_ylabel('Residual')

plt.tight_layout()
plt.show()

### Stationarity Testing

**Stationary Series:** Mean, variance, autocorrelation constant over time

**Why Important:** Most models require stationary data

**Tests:**
- ADF (Augmented Dickey-Fuller)
- KPSS (Kwiatkowski-Phillips-Schmidt-Shin)

In [ ]:
from statsmodels.tsa.stattools import adfuller, kpss

def check_stationarity(timeseries, title='Time Series'):
    """Check if time series is stationary"""
    
    # ADF Test
    print(f'\n{title}')
    print('=' * 50)
    adf_result = adfuller(timeseries.dropna())
    print('\nAugmented Dickey-Fuller Test:')
    print(f'ADF Statistic: {adf_result[0]:.4f}')
    print(f'p-value: {adf_result[1]:.4f}')
    print(f'Critical Values:')
    for key, value in adf_result[4].items():
        print(f'  {key}: {value:.4f}')
    
    if adf_result[1] <= 0.05:
        print('Result: Stationary (reject H0)')
    else:
        print('Result: Non-stationary (fail to reject H0)')
    
    # KPSS Test
    kpss_result = kpss(timeseries.dropna())
    print('\nKPSS Test:')
    print(f'KPSS Statistic: {kpss_result[0]:.4f}')
    print(f'p-value: {kpss_result[1]:.4f}')
    
    if kpss_result[1] <= 0.05:
        print('Result: Non-stationary (reject H0)')
    else:
        print('Result: Stationary (fail to reject H0)')

check_stationarity(ts, 'Original Series')

### Making Series Stationary

**Methods:**
1. **Differencing:** Remove trend
2. **Log transform:** Stabilize variance
3. **Detrending:** Remove trend component

In [ ]:
# 1. Differencing
ts_diff = ts.diff().dropna()

# 2. Log + Differencing
ts_log = np.log(ts)
ts_log_diff = ts_log.diff().dropna()

# 3. Seasonal Differencing
ts_seasonal_diff = ts.diff(12).dropna()  # lag=12 for monthly seasonality

# Plot transformations
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

ts.plot(ax=axes[0, 0], title='Original')
ts_diff.plot(ax=axes[0, 1], title='First Difference')
ts_log.plot(ax=axes[1, 0], title='Log Transform')
ts_log_diff.plot(ax=axes[1, 1], title='Log + Difference')

plt.tight_layout()
plt.show()

# Check stationarity after differencing
check_stationarity(ts_diff, 'After First Differencing')

### ACF and PACF

**ACF (Autocorrelation Function):** Correlation with lagged values

**PACF (Partial Autocorrelation Function):** Direct correlation (removes indirect effects)

**Use:** Determine AR and MA orders for ARIMA

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# ACF
plot_acf(ts_diff, lags=40, ax=axes[0])
axes[0].set_title('Autocorrelation Function (ACF)')

# PACF
plot_pacf(ts_diff, lags=40, ax=axes[1])
axes[1].set_title('Partial Autocorrelation Function (PACF)')

plt.tight_layout()
plt.show()

print("""
Interpretation:
- ACF cuts off after lag q → MA(q)
- PACF cuts off after lag p → AR(p)
- Both tail off → ARMA(p,q)
""")

### Lag Features

**Purpose:** Create features from past values for ML models

In [ ]:
# Create lag features
df = pd.DataFrame({'value': ts})

# Add lag features
for lag in range(1, 8):
    df[f'lag_{lag}'] = df['value'].shift(lag)

# Rolling statistics
df['rolling_mean_7'] = df['value'].rolling(window=7).mean()
df['rolling_std_7'] = df['value'].rolling(window=7).std()
df['rolling_mean_30'] = df['value'].rolling(window=30).mean()

# Time-based features
df['day_of_week'] = df.index.dayofweek
df['month'] = df.index.month
df['quarter'] = df.index.quarter

print(df.head(10))
print(f"\nFeatures created: {df.shape[1]}")

---
## 8.2 Traditional Time Series Methods

### ARIMA (AutoRegressive Integrated Moving Average)

**Components:**
- AR(p): Autoregressive - depends on p past values
- I(d): Integrated - differencing order
- MA(q): Moving Average - depends on q past errors

**Parameters:** ARIMA(p, d, q)
- p: Number of lag observations
- d: Degree of differencing
- q: Size of moving average window

In [ ]:
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_squared_error, mean_absolute_error

# Split data
train_size = int(len(ts) * 0.8)
train, test = ts[:train_size], ts[train_size:]

# Fit ARIMA model
model = ARIMA(train, order=(1, 1, 1))  # ARIMA(p=1, d=1, q=1)
fitted_model = model.fit()

# Summary
print(fitted_model.summary())

# Forecast
forecast = fitted_model.forecast(steps=len(test))

# Evaluate
mse = mean_squared_error(test, forecast)
mae = mean_absolute_error(test, forecast)
rmse = np.sqrt(mse)

print(f"\nMSE: {mse:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"MAE: {mae:.2f}")

# Plot
plt.figure(figsize=(14, 6))
plt.plot(train.index, train, label='Train')
plt.plot(test.index, test, label='Test', color='orange')
plt.plot(test.index, forecast, label='Forecast', color='red')
plt.legend()
plt.title('ARIMA Forecast')
plt.grid(True)
plt.show()

In [ ]:
# Auto ARIMA - automatic parameter selection
# Install: pip install pmdarima
from pmdarima import auto_arima

# Find optimal parameters
auto_model = auto_arima(
    train,
    start_p=0, start_q=0,
    max_p=5, max_q=5,
    d=None,                    # auto-detect d
    seasonal=False,
    stepwise=True,
    suppress_warnings=True,
    error_action='ignore',
    trace=True                 # print search progress
)

print(f"\nBest ARIMA order: {auto_model.order}")
print(auto_model.summary())

# Forecast
forecast_auto = auto_model.predict(n_periods=len(test))
print(f"\nAuto ARIMA RMSE: {np.sqrt(mean_squared_error(test, forecast_auto)):.2f}")

### SARIMA (Seasonal ARIMA)

**What:** ARIMA with seasonal components

**Parameters:** SARIMA(p,d,q)(P,D,Q,s)
- (p,d,q): Non-seasonal parameters
- (P,D,Q,s): Seasonal parameters
- s: Seasonal period (12 for monthly, 4 for quarterly)

In [ ]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

# SARIMA model
sarima_model = SARIMAX(
    train,
    order=(1, 1, 1),           # non-seasonal (p,d,q)
    seasonal_order=(1, 1, 1, 12)  # seasonal (P,D,Q,s)
)

sarima_fitted = sarima_model.fit(disp=False)
print(sarima_fitted.summary())

# Forecast
sarima_forecast = sarima_fitted.forecast(steps=len(test))

# Evaluate
print(f"\nSARIMA RMSE: {np.sqrt(mean_squared_error(test, sarima_forecast)):.2f}")

# Plot
plt.figure(figsize=(14, 6))
plt.plot(train.index, train, label='Train')
plt.plot(test.index, test, label='Test', color='orange')
plt.plot(test.index, sarima_forecast, label='SARIMA Forecast', color='red')
plt.legend()
plt.title('SARIMA Forecast')
plt.grid(True)
plt.show()

### Exponential Smoothing & Holt-Winters

**Types:**
1. **Simple Exponential Smoothing:** Level only
2. **Holt's Linear:** Level + Trend
3. **Holt-Winters:** Level + Trend + Seasonality

In [ ]:
from statsmodels.tsa.holtwinters import ExponentialSmoothing

# Holt-Winters model
hw_model = ExponentialSmoothing(
    train,
    trend='add',              # 'add' or 'mul'
    seasonal='add',           # 'add' or 'mul'
    seasonal_periods=30       # period length
)

hw_fitted = hw_model.fit()
hw_forecast = hw_fitted.forecast(steps=len(test))

print(f"Holt-Winters RMSE: {np.sqrt(mean_squared_error(test, hw_forecast)):.2f}")

# Plot
plt.figure(figsize=(14, 6))
plt.plot(train.index, train, label='Train')
plt.plot(test.index, test, label='Test', color='orange')
plt.plot(test.index, hw_forecast, label='Holt-Winters', color='red')
plt.legend()
plt.title('Holt-Winters Forecast')
plt.grid(True)
plt.show()

---
## 8.3 Modern Time Series with ML

### Prophet (by Facebook)

**What:** Automatic forecasting tool for business time series

**Features:**
- Handles missing data
- Automatic seasonality detection
- Handles outliers
- Holidays and events
- Easy to use

**Best for:** Business metrics, daily observations

In [ ]:
# Install: pip install prophet
from prophet import Prophet

# Prepare data (Prophet requires 'ds' and 'y' columns)
df_prophet = pd.DataFrame({
    'ds': ts.index,
    'y': ts.values
})

train_prophet = df_prophet[:train_size]
test_prophet = df_prophet[train_size:]

# Create and fit model
prophet_model = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=True,
    daily_seasonality=False,
    seasonality_mode='additive',  # or 'multiplicative'
    changepoint_prior_scale=0.05   # flexibility of trend (0.001-0.5)
)

prophet_model.fit(train_prophet)

# Make future dataframe
future = prophet_model.make_future_dataframe(periods=len(test), freq='D')
forecast = prophet_model.predict(future)

# Extract predictions for test period
prophet_pred = forecast.iloc[train_size:]['yhat'].values

print(f"Prophet RMSE: {np.sqrt(mean_squared_error(test, prophet_pred)):.2f}")

# Plot forecast
fig = prophet_model.plot(forecast)
plt.title('Prophet Forecast')
plt.show()

# Plot components
fig = prophet_model.plot_components(forecast)
plt.show()

In [ ]:
# Add custom seasonalities and holidays
prophet_custom = Prophet()

# Add custom seasonality
prophet_custom.add_seasonality(
    name='monthly',
    period=30.5,
    fourier_order=5
)

# Add country holidays
prophet_custom.add_country_holidays(country_name='US')

prophet_custom.fit(train_prophet)
forecast_custom = prophet_custom.predict(future)

### LSTM for Time Series

**What:** Deep learning approach using LSTM (Long Short-Term Memory)

**Advantages:**
- Captures long-term dependencies
- Handles non-linear patterns
- Multivariate forecasting

**When to use:** Complex patterns, large datasets

In [ ]:
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from sklearn.preprocessing import MinMaxScaler

# Prepare data for LSTM
def create_sequences(data, seq_length):
    """Create sequences for LSTM"""
    X, y = [], []
    for i in range(len(data) - seq_length):
        X.append(data[i:i+seq_length])
        y.append(data[i+seq_length])
    return np.array(X), np.array(y)

# Scale data
scaler = MinMaxScaler()
ts_scaled = scaler.fit_transform(ts.values.reshape(-1, 1))

# Create sequences
seq_length = 30  # use 30 days to predict next day
X, y = create_sequences(ts_scaled, seq_length)

# Split
train_size = int(len(X) * 0.8)
X_train, X_test = X[:train_size], X[train_size:]
y_train, y_test = y[:train_size], y[train_size:]

# Reshape for LSTM [samples, timesteps, features]
X_train = X_train.reshape((X_train.shape[0], X_train.shape[1], 1))
X_test = X_test.reshape((X_test.shape[0], X_test.shape[1], 1))

# Build LSTM model
model = Sequential([
    LSTM(50, activation='relu', return_sequences=True, input_shape=(seq_length, 1)),
    Dropout(0.2),
    LSTM(50, activation='relu'),
    Dropout(0.2),
    Dense(1)
])

model.compile(optimizer='adam', loss='mse')
print(model.summary())

# Train
history = model.fit(
    X_train, y_train,
    epochs=50,
    batch_size=32,
    validation_split=0.1,
    verbose=0
)

# Predict
y_pred = model.predict(X_test)

# Inverse transform
y_test_inv = scaler.inverse_transform(y_test)
y_pred_inv = scaler.inverse_transform(y_pred)

# Evaluate
lstm_rmse = np.sqrt(mean_squared_error(y_test_inv, y_pred_inv))
print(f"\nLSTM RMSE: {lstm_rmse:.2f}")

# Plot training history
plt.figure(figsize=(14, 5))
plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.title('Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

# Plot predictions
plt.subplot(1, 2, 2)
plt.plot(y_test_inv, label='Actual', alpha=0.7)
plt.plot(y_pred_inv, label='Predicted', alpha=0.7)
plt.title('LSTM Predictions')
plt.xlabel('Time')
plt.ylabel('Value')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

### Multi-variate Time Series

**What:** Multiple input features to predict target

In [ ]:
# Create multivariate dataset
np.random.seed(42)
dates = pd.date_range('2020-01-01', periods=365, freq='D')

df_multi = pd.DataFrame({
    'feature1': np.random.randn(365).cumsum() + 100,
    'feature2': np.random.randn(365).cumsum() + 50,
    'feature3': np.sin(np.arange(365) * 2 * np.pi / 365) * 10,
    'target': np.random.randn(365).cumsum() + 200
}, index=dates)

# Prepare sequences
def create_multivariate_sequences(data, seq_length):
    X, y = [], []
    for i in range(len(data) - seq_length):
        X.append(data[i:i+seq_length, :-1])  # all features except target
        y.append(data[i+seq_length, -1])     # target only
    return np.array(X), np.array(y)

# Scale
scaler_multi = MinMaxScaler()
data_scaled = scaler_multi.fit_transform(df_multi.values)

# Create sequences
seq_length = 30
X, y = create_multivariate_sequences(data_scaled, seq_length)

# Split
train_size = int(len(X) * 0.8)
X_train, X_test = X[:train_size], X[train_size:]
y_train, y_test = y[:train_size], y[train_size:]

# Build model
model_multi = Sequential([
    LSTM(64, activation='relu', return_sequences=True, 
         input_shape=(seq_length, X.shape[2])),
    Dropout(0.2),
    LSTM(32, activation='relu'),
    Dropout(0.2),
    Dense(1)
])

model_multi.compile(optimizer='adam', loss='mse')

# Train
history = model_multi.fit(
    X_train, y_train,
    epochs=30,
    batch_size=32,
    validation_split=0.1,
    verbose=0
)

# Predict
y_pred = model_multi.predict(X_test)
print(f"Multivariate LSTM RMSE: {np.sqrt(mean_squared_error(y_test, y_pred)):.4f}")

---
### Model Comparison

| Method | Complexity | Data Need | Interpretability | Best Use Case |
|--------|------------|-----------|------------------|---------------|
| ARIMA | Low | Small | High | Stationary, short-term |
| SARIMA | Medium | Small-Medium | High | Seasonal patterns |
| Holt-Winters | Low | Small | High | Trend + Seasonality |
| Prophet | Low | Medium | Medium | Business metrics, holidays |
| LSTM | High | Large | Low | Complex, non-linear, long-term |
| XGBoost | Medium | Medium | Medium | Feature engineering, tabular |

---
### Quick Reference Guide

**Data Preparation:**
1. Check for missing values
2. Test stationarity (ADF/KPSS)
3. Make stationary (differencing, log)
4. Check ACF/PACF for AR/MA orders

**Model Selection:**
- Simple patterns: ARIMA, Holt-Winters
- Seasonal data: SARIMA, Prophet
- Business data: Prophet
- Complex patterns: LSTM, XGBoost
- Multiple features: Multivariate LSTM

**Evaluation Metrics:**
- MAE: Mean Absolute Error
- RMSE: Root Mean Squared Error
- MAPE: Mean Absolute Percentage Error
- SMAPE: Symmetric MAPE

**Best Practices:**
- Use walk-forward validation (not random split)
- Check residuals for white noise
- Consider ensemble of multiple models
- Monitor forecast horizon (accuracy decreases)
- Retrain regularly with new data

---
[← Back to Index](Index.ipynb)